
<div style="max-width: 1000px; margin: 0 auto; font-family: sans-serif;">

<div style="background: #02A36F; color: white; border-radius: 8px; padding: 28px 32px; text-align: center; position: relative;">
  <div style="font-size: 14pt; font-weight: 600; text-transform: uppercase; letter-spacing: 1px; opacity: 0.85; margin-bottom: 8px;">Practice</div>
  <div style="font-size: 24pt; font-weight: 700; line-height: 1.3;">Load Data Incrementally</div>
  <div style="font-size: 14pt; margin-top: 12px; opacity: 0.9;">Practice creating an empty table and loading data into it with COPY INTO.</div>
</div>

</div>

### Instructions

In Lesson 5, you used COPY INTO to load both CSV files into `current_employees_copyinto`. Now you'll practice the same pattern from scratch: create an empty table, load data, and prove it's idempotent.

Your goal: create a table called **`practice_copyinto`**, load the CSV files into it, and verify the behavior.

### Task 1: Create an empty table

Create a table called `practice_copyinto` with 4 columns: `ID` (INT), `FirstName` (STRING), `Country` (STRING), `Role` (STRING).

In [0]:
%sql
CREATE TABLE practice_copyinto (
    ID INT,
    FirstName STRING,
    Country STRING,
    Role STRING
);

<details>
<summary>Hint</summary>

<pre><code>CREATE TABLE IF NOT EXISTS practice_copyinto (
  ID INT,
  FirstName STRING,
  Country STRING,
  Role STRING
);</code></pre>

</details>

Verify the table is empty.

In [0]:
%sql
SELECT COUNT(*) AS row_count FROM practice_copyinto;

row_count
0


### Task 2: Load data with COPY INTO

Use COPY INTO to load all CSV files from the `myfiles` volume into `practice_copyinto`. Use the Python cell below (COPY INTO requires `spark.sql` in notebooks).

In [0]:
result = spark.sql("""
    COPY INTO practice_copyinto
    FROM '/Volumes/dbacademy/get_started_de/myfiles/'
    FILEFORMAT = CSV
    FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true')
""")

result.display()

num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
6,6,0


<details>
<summary>Hint</summary>

<pre><code>result = spark.sql("""
    COPY INTO practice_copyinto
    FROM '/Volumes/dbacademy/get_started_de/myfiles/'
    FILEFORMAT = CSV
    FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true')
""")
result.display()</code></pre>

</details>

You should see **num_affected_rows: 6** (4 from `employees.csv` + 2 from `employees2.csv`).

### Task 3: Verify the data

Query the table to confirm all 6 rows loaded.

In [0]:
%sql
SELECT *
FROM practice_copyinto
ORDER BY ID;

ID,FirstName,Country,Role
1,Kristi,United States,Data Engineer
2,Sophia,Germany,Data Analyst
3,Peter,Canada,ML Engineer
4,Zebi,India,Data Scientist
5,Maria,Brazil,Data Engineer
6,Aiden,Australia,Data Analyst


<details>
<summary>Hint</summary>

<pre><code>SELECT * FROM practice_copyinto;</code></pre>

</details>

### Task 4: Prove idempotency

Run the exact same COPY INTO command again. How many rows get loaded this time?

In [0]:
result = spark.sql("""
    COPY INTO practice_copyinto
    FROM '/Volumes/dbacademy/get_started_de/myfiles/'
    FILEFORMAT = CSV
    FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true')
""")
result.display()

num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
0,0,0


**num_affected_rows: 0**. The files were already processed. No duplicates.

### Task 5: Check the version history

Use `DESCRIBE HISTORY` to confirm that only the initial load created a new version, not the re-run.

In [0]:
%sql
DESCRIBE HISTORY practice_copyinto;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
1,2026-10-19T10:03:58.000Z,76258775815584,vaibhavmaheshwari5141@gmail.com,COPY INTO,Map(statsOnLoad -> true),null,List(1412118294671753),ee203981-08e8-4031-bcc2-cfa51b2d8c71,1019-091718-b4km7bky-v2n,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 6, numOutputBytes -> 1699, numSkippedCorruptFiles -> 0)",null,Databricks-Runtime/19.9.x-aarch64-photon-scala2.13
0,2026-10-19T10:02:19.000Z,76258775815584,vaibhavmaheshwari5141@gmail.com,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-6167bb67-fc89-4dcd-8232-6f3f70dac5ec"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-3f156fde-fe31-4f35-8e8f-91df9852eded""}, statsOnLoad -> false)",null,List(1412118294671753),2fee96a8-8a34-4234-833e-fb9ceea2c40e,1019-091718-b4km7bky-v2n,null,WriteSerializable,true,Map(),null,Databricks-Runtime/19.9.x-aarch64-photon-scala2.13


<details>
<summary>Hint</summary>

<pre><code>DESCRIBE HISTORY practice_copyinto;</code></pre>

</details>


<div style="max-width: 900px; margin: 0 auto; font-family: sans-serif;">
<div style="margin-top: 10px; padding: 18px 24px; background: #FFF6F4; border: 3px solid #FF5F46; border-radius: 10px;">
  <div style="font-size: 14pt; color: #0b2026; line-height: 1.6;">
    <div style="font-weight: 700; margin-bottom: 8px;">Nice work! You just:</div>
    <ul style="padding-left: 20px; margin: 0;">
      <li>Created an empty table with a defined schema</li>
      <li>Loaded data incrementally with <code>COPY INTO</code></li>
      <li>Proved idempotency by re-running and getting 0 rows</li>
      <li>Verified the transaction log only records actual data changes</li>
    </ul>
  </div>
</div>
</div>


<!-- CHECKPOINT: Lessons 4-5 -->

<div style="max-width: 900px; margin: 0 auto; font-family: sans-serif;">

<div style="background: #1B5162; color: white; border-radius: 8px; padding: 24px 28px; text-align: center;">
  <div style="font-size: 14pt; font-weight: 600; text-transform: uppercase; letter-spacing: 1px; opacity: 0.85; margin-bottom: 8px;">Checkpoint</div>
  <div style="font-size: 20pt; font-weight: 700;">What You've Done So Far</div>
</div>

<div style="margin-top: 16px; padding: 20px 24px; background: #F9F7F4; border-radius: 8px; box-shadow: 0 2px 8px rgba(27,49,57,0.06);">
  <div style="font-size: 14pt; color: #0b2026; line-height: 1.7;">
    <p>Over Lessons 4 and 5, you learned three ingestion methods:</p>
    <ul style="padding-left: 20px; margin: 8px 0;">
      <li><strong>CTAS</strong> — code-driven, one-time table creation with full control over columns and format options</li>
      <li><strong>Upload UI</strong> — manual drag-and-drop for quick, ad-hoc imports</li>
      <li><strong>COPY INTO</strong> — incremental loading that's safe to re-run because it tracks which files have already been processed</li>
    </ul>
    <p>Combined with Lessons 1 through 3, you can now find raw data, create tables, modify their contents, audit every change, and choose the right ingestion method for the job.</p>
  </div>
</div>

<div style="margin-top: 16px; padding: 16px 20px; background: #F8F9FC; border-left: 4px solid #1B5162; border-radius: 6px;">
  <div style="font-size: 14pt; color: #0b2026; line-height: 1.6;">
    <strong>Quick self-check:</strong> If a teammate asked "what's the difference between CTAS and COPY INTO?" could you explain when you'd use each one?
  </div>
</div>